# Notebook complémentaire — Actualisation et points de bascule

Ce notebook utilise `DICE.py` et développe deux thèmes : le facteur d’actualisation et les points de bascule climatiques.

## Programme
- Charger DICE et simuler une référence.
- Relier facteur d’actualisation et taux d’intérêt.
- Comparer trois valeurs de `p.rho`, dont celle proposée par Stern.
- Calculer les politiques optimales et comparer réduction des émissions, taxe carbone, dommages et température.
- Étudier des fonctions de dommages non linéaires inspirées de Weitzman et un seuil à 3 °C.


## 0) Charger `DICE.py`

Placez `DICE.py` à côté du notebook ou adaptez le chemin d’importation.


In [ ]:
# Exécutez cette cellule une fois pour vérifier/installer les paquets Python requis pour ce portable.

import sys
import subprocess
import importlib.util

required = {
    "matplotlib": "matplotlib",
    "numba": "numba",
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "tqdm": "tqdm",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing]
    )

print("Python environment ready.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from DICE import (
    Params,
    init_states,
    update_path,
    mat_to_df,
    obj_fun,
    run_optimal_policy,
)

# Initialisation de base
p       = Params()
sim     = init_states(p)
timevec = range(1, p.nT)
sim     = update_path(sim, timevec, p)

# Peek à l'image de données si vous voulez
df = mat_to_df(sim, p)
df.head()


## 1) Controverse sur l’actualisation

Le taux de préférence pure pour le présent $\rho$ détermine le poids accordé aux générations futures. Un $\rho$ élevé réduit généralement l’effort climatique immédiat ; un $\rho$ faible, comme chez Stern, conduit à une action plus précoce et plus forte.

- **Nordhaus** utilise souvent $\rho\simeq1{,}5\%$ par an dans DICE.
- **Stern** retient un taux proche de zéro, $\rho\simeq0{,}1\%$.

[Rapport Stern](https://webarchive.nationalarchives.gov.uk/ukgwa/20100407172811/http://www.hm-treasury.gov.uk/stern_review_report.htm)

Le débat est autant éthique que technique : quel poids donner au bien-être des générations futures ?


### 1-A) Du facteur d’actualisation au taux d’intérêt

Le facteur d’actualisation est $\beta=1/(1+\rho)$. Sous des hypothèses usuelles, le taux d’intérêt réel vérifie approximativement
$$r\simeq\rho+\gamma g,$$
où $g$ est la croissance de la consommation par habitant et $\gamma$ l’aversion relative au risque.

**Exercice**
1. Lisez `p.rho` et `p.gamma`.
2. Supposez $g=2\%$.
3. Calculez $\beta$ et $r$, puis testez d’autres valeurs de $g$.


In [ ]:
rho = p.rho
gamma = p.gamma
g = 0.02

# Calculer les deux quantités demandées:
# beta = 1 / (1 + rho)
# r = rho + gamma * g


### 1-B) Choisir trois valeurs de $\rho$

Les taux d’intérêt réels de long terme se situent souvent entre 1 % et 4 % par an. Consultez par exemple [FRED](https://fred.stlouisfed.org/series/IRLTLT01USA156N) et [CEPR VoxEU](https://cepr.org/voxeu/columns/global-r).

Comparez :
- Nordhaus : $\rho=0{,}015$ ;
- valeur intermédiaire : $\rho=0{,}005$ ;
- Stern : $\rho=0{,}001$.

Créez un étalonnage pour chaque valeur.


In [ ]:
# >>> Your code here <<<
pNordhaus     = Params()
pIntermediate = Params()
pStern        = Params()
# une torsion pour obtenir le calcul gérable, sinon les fenêtres de planificateur s'étendent beaucoup
pIntermediate.toly   = 0.01
pStern.toly          = 0.01
# ...

### 1-C) Calculer la politique optimale

Pour réduire le temps de calcul, optimisez seulement le taux de réduction des émissions $\mu_t$.


In [ ]:
# >>> Your code here <<<
bounds_s   = [(0, 1)]
control_id = [pNordhaus.i_mu]
path_opt_Nordhaus = run_optimal_policy(sim.copy(), timevec, pNordhaus, bounds_s, control_id)
# ...

### 1-D) Comparer réduction, taxe carbone, dommages et température

Tracez ces quatre variables pour les trois scénarios d’actualisation.


In [ ]:
# >>> Your code here <<<

plt.subplot(1, 4, 1)
# réduction des parcelles
# var name: "mu"

plt.subplot(1, 4, 2)
# dommages matériels
# damages = p.a2 * (sim[0:,p.i_T_AT] ** p.a3)

plt.subplot(1, 4, 3)
# graphique taxe carbone
# var name: "Tax"

plt.subplot(1, 4, 4)
# température de la parcelle
# var name: "T_AT"

plt.show()


### 1-E) Interpréter

Comment une baisse de $\rho$ modifie-t-elle la réduction des émissions, la taxe carbone, les températures et les dommages ? Les résultats correspondent-ils à l’intuition du cours ?


> Vous avez écrit la réponse ici.

## 2) Rôle des points de bascule

Les points de bascule sont des changements brusques et potentiellement irréversibles du système climatique : fonte des calottes, modification de l’AMOC ou dégel du pergélisol. Ils créent des risques non linéaires et à queues épaisses que des dommages quadratiques lisses peuvent sous-estimer.


### 2-A) Doubler les dommages quadratiques

La référence utilise $D(T)=a_2T^{a_3}$. Créez `pB = Params()` pour la référence et `pN = Params()` pour l’alternative, puis fixez `pN.a2 = 2 * pN.a2`. Initialisez et optimisez les deux trajectoires avec les mêmes bornes et le même horizon.


In [ ]:
pB = Params()
pN = Params()
pN.a2 = 2 * pN.a2

timevec = range(1, p.nT)
simB = init_states(pB)
simN = init_states(pN)
bounds_mu = (0, 1)
control_id = [pB.i_mu]

# Optimiser simB avec pB et simN avec pN à l’aide de run_optimal_policy.


### 2-B) Comparer les résultats

Comparez la référence et les dommages doublés : dommages, taxe carbone, taux de réduction des émissions et température.


In [ ]:
# Comparer simB (référence) avec simN (a2 doublé) en quatre panneaux.
# Utilisez les colonnes i_mu, i_Tax et i_T_AT.
# Calculer les parts de dommages comme p.a2 * température ** p.a3 pour chaque étalonnage.


### 2-C) Interpréter

Expliquez comment des dommages plus élevés modifient la taxe, la réduction des émissions et la température au cours du temps.


> Vous avez écrit la réponse ici.

### 2-D) Dommages à la Weitzman et queues épaisses

Utilisez le second terme déjà codé dans `DICE.py` :
$$D_W(T)=a_2T^{a_3}+a_4T^{a_5}.$$

Définissez :
```python
pW.a4 = 5.0703e-06
pW.a5 = 6.754
pW.a6 = 0.0
```

Avec `a6=0`, le terme supplémentaire est actif pour toute température positive. Simulez la politique optimale et le laisser-faire.


In [ ]:
pW = Params()
pW.a4 = 5.0703e-06
pW.a5 = 6.754
pW.a6 = 0.0

# Initialisez un chemin Weitzman optimal et une copie laissez-faire.
# Optimiser le premier avec run_optimal_policy et propager le second avec update_path.


### 2-E) Comparer Nordhaus et Weitzman

Comparez `simB`, `simW` et `simW_LF` : réduction, dommages, taxe carbone et température. Pour Weitzman, incluez les deux termes `a2*T**a3` et `a4*T**a5`.


In [ ]:
# Utilisez simB, simW et simW LF depuis les cellules précédentes.
# Construire quatre panneaux pour i_mu, dommages, i_Tax et i_T_AT.


### 2-F) Interpréter

Les queues plus épaisses conduisent-elles à une taxe et à une réduction plus fortes ou plus précoces ? Testez plusieurs valeurs du coefficient de dommage.


> Vous avez écrit la réponse ici.

### 2-G) Introduire un seuil à 3 °C

Pour doubler les dommages quadratiques au-dessus de 3 °C, définissez :
```python
pG.a4 = pG.a2
pG.a5 = pG.a3
pG.a6 = 3.0
```

Optimisez cet étalonnage, construisez son équivalent en laisser-faire, puis comparez-les à la référence.


In [ ]:
pG = Params()
pG.a4 = pG.a2
pG.a5 = pG.a3
pG.a6 = 3.0

# Initialiser simG et simG LF, optimiser simG et propager simG LF.
# Alors comparez-les avec simB.


> Vous avez écrit la réponse ici.